In [1]:
!pip uninstall -y transformers
!pip install "transformers==4.50.0"

Found existing installation: transformers 5.16.1
Uninstalling transformers-5.16.1:
  Successfully uninstalled transformers-5.16.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which i

In [2]:
import transformers
print(transformers.__version__)

4.50.0


GPT-1

In [4]:
import numpy as np
import tensorflow as tf
from transformers import TFAutoModelForCausalLM,AutoTokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/openai-gpt")
model = TFAutoModelForCausalLM.from_pretrained("openai-community/openai-gpt")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/479M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFOpenAIGPTLMHeadModel: ['h.7.attn.bias', 'h.6.attn.bias', 'h.2.attn.bias', 'h.0.attn.bias', 'h.5.attn.bias', 'h.3.attn.bias', 'h.4.attn.bias', 'h.1.attn.bias', 'h.9.attn.bias', 'h.10.attn.bias', 'h.11.attn.bias', 'h.8.attn.bias']
- This IS expected if you are initializing TFOpenAIGPTLMHeadModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFOpenAIGPTLMHeadModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFOpenAIGPTLMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFOpenAIGPTLMHeadModel for predicti

In [8]:
if(tokenizer.pad_token is None):
  tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [25]:
prompt =  "the old house stood at the end of the"
inputs = tokenizer(prompt,return_tensors = "tf")
print(inputs["input_ids"])
print(inputs["attention_mask"])

tf.Tensor([[ 481 1122 1114 1137  491  481  872  498  481]], shape=(1, 9), dtype=int32)
tf.Tensor([[1 1 1 1 1 1 1 1 1]], shape=(1, 9), dtype=int32)


In [28]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].numpy().tolist())
print(tokens)

['the</w>', 'old</w>', 'house</w>', 'stood</w>', 'at</w>', 'the</w>', 'end</w>', 'of</w>', 'the</w>']


In [27]:
#Greedy Decoding
greedy_output = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 50,
    do_sample = False,
    pad_token_id = tokenizer.pad_token_id
)

greedy_text = tokenizer.decode(greedy_output[0],skip_special_tokens=True)
print(greedy_text)

the old house stood at the end of the street , and the house was a large , rambling , two - story structure with a large front porch . the front door was open , and a man was standing on the porch , smoking a cigarette . 
 " hello , " he said . 
 "


In [29]:
#Temperature
temperature_output = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens=50,
    do_sample = True,
    temperature=0.7,
    pad_token_id = tokenizer.pad_token_id
)

temperature_text = tokenizer.decode(temperature_output[0],skip_special_tokens=True)
print(temperature_text)

the old house stood at the end of the street . 
 i had no idea how much time had passed , but i felt as if time had stopped . 
 i heard a car drive up and then the sound of a door opening , then shutting . 
 " it 's okay . " i said


In [30]:
#Top-k sampling
topk_output = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens=50,
    do_sample = True,
    temperature=0.7,
    top_k=50,
    pad_token_id = tokenizer.pad_token_id
)

topk_text = tokenizer.decode(topk_output[0],skip_special_tokens=True)
print(topk_text)

the old house stood at the end of the road . he felt a shiver run through him , but no one else seemed to notice . he was about to turn and go back to the car when he saw a woman standing at the edge of the driveway , staring into the darkness . she seemed to


In [31]:
topp_output = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 50,
    do_sample = True,
    temperature = 0.7,
    top_p = 0.85,
    pad_token_id = tokenizer.pad_token_id
)

topp_text = tokenizer.decode(topp_output[0],skip_special_tokens=True)
print(topp_text)

the old house stood at the end of the driveway . 
 " it 's the old house . " she smiled . " it 's been there since i was a child . " 
 " i 'm sorry i was n't there when you got married , " he said . 
 " do n't be .
